In [1]:
import pandas as pd
import pypsa
import os


In [2]:
n = pypsa.Network()
n.set_snapshots(pd.date_range("2026-01-01 10:00:00", "2026-01-01 12:00:00", freq="h"))

In [3]:
n.snapshots

DatetimeIndex(['2026-01-01 10:00:00', '2026-01-01 11:00:00',
               '2026-01-01 12:00:00'],
              dtype='datetime64[us]', name='snapshot', freq='h')

In [4]:
buses_df = pd.read_excel(os.path.join(os.getcwd(), "Data", "buses.xlsx"))
buses_df.set_index('name', inplace=True) #name has to be set as the index, for bulk imports to work
# buses_df.index = n.snapshots
n.add('Bus', buses_df.index, v_nom=buses_df['v_nom'], x=buses_df['x'], y=buses_df['y'], carrier=buses_df['carrier'])

In [5]:
trafos_df = pd.read_excel(os.path.join(os.getcwd(), "Data", "link_trafo.xlsx"), index_col=0)
n.add('Link', trafos_df.index, bus0=trafos_df['bus0'], bus1=trafos_df['bus1'], p_nom=trafos_df['p_nom'], efficiency=trafos_df['efficiency'])

In [6]:
gens_df = pd.read_excel(os.path.join(os.getcwd(), "Data", "gens.xlsx"), index_col=0)
gens_pu_df = pd.read_excel(os.path.join(os.getcwd(), "Data", "gen_profile_pu.xlsx"), index_col=0)
gens_pu_df.index = pd.to_datetime(gens_pu_df.index).round("h") #floating point error, excel saves as float, which would read exactly by pandas, and not rounded-off like excel does.
# solar_pu = gens_pu_df.loc['solar']
n.add('Generator', gens_df.index, bus=gens_df['bus'], p_nom=gens_df['p_nom'], marginal_cost=gens_df['marginal_cost'], p_max_pu=gens_pu_df)

In [7]:
loads_df = pd.read_excel(os.path.join(os.getcwd(), "Data", "loads.xlsx"), index_col=0)
loads_pu_df = pd.read_excel(os.path.join(os.getcwd(), "Data", "load_profiles_pu.xlsx"), index_col=0)
loads_pu_df.index = pd.to_datetime(loads_pu_df.index).round("h") 

industry_1_peak = 500 #MW
residential_1_peak = 20 #MW
kritis_1_peak = 50 #MW


industry_1_profile = loads_pu_df.loc[:,'Industry'] * industry_1_peak
residential_1_profile = loads_pu_df.loc[:,'Residential'] * residential_1_peak
kritis_1_profile = loads_pu_df.loc[:,'Kritis'] * kritis_1_peak

p_set_df = pd.DataFrame({'industry': industry_1_profile, 'residential': residential_1_profile, 'kritis': kritis_1_profile})
p_set_df = p_set_df[loads_df.index] #reorder the columns to match the loads_df index

n.add('Load', loads_df.index, bus=loads_df['bus'], p_set=p_set_df)

# n.add

In [8]:
n.consistency_check()

Index(['Industry_1', 'Kritis_1', 'Residential_1', 'Extern_grid', 'MV_bus_1',
       'MV_bus_2'],
      dtype='object', name='name')
Index(['substation_1', 'substation_2', 'industry', 'kritis', 'residential'], dtype='object', name='name')


In [9]:
print(n.model)

None


In [10]:
n.add('Carrier', 'DC', co2_emissions=0, color='black')

In [11]:
# n.lpf()
n.optimize(),
n.objective

C:\Users\student\AppData\Local\Temp\ipykernel_17944\2373778960.py:2: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n.optimize(),
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.67s
Status: warning
Termination condition: infeasible
Solution: 0 primals, 0 duals
Objective: nan
Solver model: available
Solver message: Infeasible



In [12]:
n.generators_t.p

name
snapshot
2026-01-01 10:00:00
2026-01-01 11:00:00
2026-01-01 12:00:00


In [13]:
n.generators

,bus,control,type,p_nom,p_nom_mod,p_nom_extendable,p_nom_min,p_nom_max,p_nom_set,p_min_pu,...,min_up_time,min_down_time,up_time_before,down_time_before,ramp_limit_up,ramp_limit_down,ramp_limit_start_up,ramp_limit_shut_down,weight,p_nom_opt
name,,,,,,,,,,,,,,,,,,,,,
Roof_pv,Residential_1,PQ,,1.0,0.0,False,0.0,inf,NaN,0.0,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,0.0
Grid,Extern_grid,PQ,,100000.0,0.0,False,0.0,inf,NaN,0.0,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,0.0


In [14]:
n.model.objective

Objective:
----------
LinearExpression: +0 Generator-p[2026-01-01 10:00:00, Roof_pv] + 0 Generator-p[2026-01-01 11:00:00, Roof_pv] + 0 Generator-p[2026-01-01 12:00:00, Roof_pv] + 50 Generator-p[2026-01-01 10:00:00, Grid] + 50 Generator-p[2026-01-01 11:00:00, Grid] + 50 Generator-p[2026-01-01 12:00:00, Grid]
Sense: min
Value: nan

In [15]:
n.loads_t.p

name
snapshot
2026-01-01 10:00:00
2026-01-01 11:00:00
2026-01-01 12:00:00


In [16]:
n.statistics().groupby(level=0).sum()

,Optimal Capacity,Installed Capacity,Supply,Withdrawal,Energy Balance,Transmission,Capacity Factor,Curtailment,Capital Expenditure,Operational Expenditure,Revenue
Generator,0.0,100001.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Link,0.0,730.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [17]:
n.branches()

active   b  b_pu  build_year         bus0  \
component name                                                      
Link      substation_1    True NaN   NaN           0  Extern_grid   
          substation_2    True NaN   NaN           0  Extern_grid   
          industry        True NaN   NaN           0     MV_bus_1   
          kritis          True NaN   NaN           0     MV_bus_1   
          residential     True NaN   NaN           0     MV_bus_1   

                                 bus1  capital_cost carrier committable  \
component name                                                            
Link      substation_1       MV_bus_1           0.0      DC       False   
          substation_2       MV_bus_2           0.0      DC       False   
          industry         Industry_1           0.0      DC       False   
          kritis             Kritis_1           0.0      DC       False   
          residential   Residential_1           0.0      DC       False   

                       cyclic_delay  ... tap_side terrain_factor  type  \
component name                       ...                                 
Link      substation_1         True  ...      NaN            1.0         
          substation_2         True  ...      NaN            1.0         
          industry             True  ...      NaN            1.0         
          kritis               True  ...      NaN            1.0         
          residential          True  ...      NaN            1.0         

                        up_time_before  v_ang_max  v_ang_min  v_nom   x  x_pu  \
component name                                                                  
Link      substation_1             1.0        NaN        NaN    NaN NaN   NaN   
          substation_2             1.0        NaN        NaN    NaN NaN   NaN   
          industry                 1.0        NaN        NaN    NaN NaN   NaN   
          kritis                   1.0        NaN        NaN    NaN NaN   NaN   
          residential              1.0        NaN        NaN    NaN NaN   NaN   

                        x_pu_eff  
component name                    
Link      substation_1       NaN  
          substation_2       NaN  
          industry           NaN  
          kritis             NaN  
          residential        NaN  

[5 rows x 75 columns]